# Base de Datos en SQL: Sistema de Gestión Académica TheBridge

**Proyecto grupal — Bootcamp Data Science y Full Stack - Grupo B**


| **Equipo** | James · Karina · Marco · Nadia · Pablo |
| **Fecha** | 10 Abril 2026 |

---

## Índice

1. [Introducción](#introduccion)
2. [Entidades del sistema](#entidades)
3. [Reglas de negocio](#reglas)
4. [Diagrama Entidad-Relación (ER)](#diagrama-er)
5. [Modelo Lógico](#modelo-logico)
6. [Conexión a la BD](#conexion)

<a id='introduccion'></a>
## 1. Introducción

Este notebook documenta el **diseño de la base de datos relacional** para el sistema de gestión académica del bootcamp **TheBridge**.

<a id='entidades'></a>
## 2. Entidades del sistema

Se identificaron las siguientes entidades principales:

| Entidad | Descripción |
|---|---|
| `alumno` | Estudiante inscrito en una promoción del bootcamp |
| `profesor` | Docente que imparte clases en una o varias promociones |
| `promocion` | Edición concreta del bootcamp (ej.: *Promoción marzo 2026 — DS*) |
| `proyecto` | Trabajo evaluable realizado por los alumnos |
| `campus` | Sede física o virtual donde se imparte la promoción |
| `vertical` | Especialidad del bootcamp (ej.: Data Science, Ciberseguridad…) |
| `modalidad` | Formato de impartición (ej.: presencial, online…) |
| `rol` | Función que desempeña un profesor en una promoción (ej.: TA, LI…) |

### Tablas intermedias (relaciones N:M)

| Tabla intermedia | Relación que resuelve |
|---|---|
| `calificacion` | Un alumno realiza muchos proyectos; un proyecto es realizado por muchos alumnos |
| `promocion_profesor` | Un profesor imparte en muchas promociones; una promoción tiene muchos profesores |

<a id='reglas'></a>
## 3. Reglas de negocio

Las siguientes reglas de negocio guiaron todas las decisiones de diseño.

**Regla 1 — Inscripción del alumno**<br>
Un alumno solo puede inscribirse en UNA promoción.
___
**Regla 2 — Composición de una promoción**<br>
Cada promoción pertenece a UN campus, UNA modalidad y UNA vertical.
Esto evita redundancia y garantiza la integridad referencial: si se cambia el nombre de un campus, basta con actualizar un único registro en `campus`.<br>
___
**Regla 3 — Profesores en una promoción**<br>
Una promoción puede tener uno o más profesores, y cada profesor tiene un ROL específico en esa promoción.<br>
Ejemplo: La promoción *marzo 2026 — DS* tiene 2 TAs y 1 LI.
La clave primaria compuesta de `promocion_profesor` es **(id_profesor, id_promocion)**.
___
**Regla 4 — Rol del profesor por promoción**<br>
Un profesor puede tener diferentes roles en distintas promociones.
Ejemplo: Un profesor puede ser TA en una promoción y LI en una promoción posterior.

Implicación en el modelo:
El rol **no es un atributo fijo del profesor**, sino un atributo de la **relación** entre profesor y promoción.  
Por eso `id_rol` vive en `promocion_profesor` y no en `profesor`.

> **Nota del equipo:** En el Modelo Lógico observaréis que la tabla `profesor` incluye `id_rol (FK)`. Esto es una simplificación válida si se asume que cada profesor tiene un rol *por defecto*, pero la regla de negocio real exige que el rol sea dinámico por promoción — lo que justifica su ubicación en `promocion_profesor`.
___
**Regla 5 — Proyectos y calificaciones**<br>
Cada alumno realiza uno o varios proyectos. Un proyecto es realizado por muchos alumnos. Por cada combinación alumno-proyecto se obtiene una calificación (nota).
La clave primaria compuesta es **(id_proyecto, id_alumno)**, garantizando que cada par alumno-proyecto tiene una única nota.


<a id='diagrama-er'></a>
## 4. Diagrama Entidad-Relación (ER)

El **Diagrama ER** es el modelo **conceptual** del sistema. Representa las entidades, sus atributos y las relaciones entre ellas, sin entrar aún en detalles de implementación.

### Convenciones usadas

| Símbolo | Significado |
|---|---|
| Rectángulo verde | Entidad |
| Elipse naranja | Clave primaria (PK) |
| Elipse amarilla | Clave foránea (FK) |
| Elipse blanca | Atributo simple |
| Rombo | Relación |
| Rectángulo azul | Tabla intermedia (resultado de relación N:M) |

### Lectura del diagrama ER

- `alumno` **inscribe** → `promocion` *(N:1)*  
- `promocion` **pertenece** → `campus` *(N:1)*  
- `promocion` **tiene** → `vertical` *(N:1)*  
- `promocion` **tiene** → `modalidad` *(N:1)*  
- `profesor` **imparte** ↔ `promocion` *(N:M)* — genera `promocion_profesor` con atributo `rol`  
- `alumno` **realiza** ↔ `proyecto` *(N:M)* — genera `calificacion` con atributo `nota`

![descripción](assets/er-diagram.png)

<a id='modelo-logico'></a>
## 5. Modelo Lógico

El **Modelo Lógico** es el paso intermedio entre el modelo conceptual (ER) y la implementación física en SQL.  
Muestra las **tablas**, sus **columnas**, las **claves primarias (PK)** y **foráneas (FK)**, y la notación de cardinalidad (pata de gallo).

![descripción](assets/entidad_relacion.png)


<a id='conexion'></a>
## 6. Conexión a la base de datos — pgAdmin 4 + Render

Una vez diseñado e implementado el modelo relacional, el equipo estableció una conexión compartida a la base de datos PostgreSQL alojada en la nube mediante **Render**, accesible para los 5 integrantes del grupo a través de **pgAdmin 4**.

### ¿Qué es Render?

[Render](https://render.com) es una plataforma cloud que permite hostear bases de datos PostgreSQL de forma gratuita. Al crear la base de datos en Render, se obtiene una **URL de conexión** con todos los parámetros necesarios (host, puerto, nombre de la BD, usuario y contraseña).

### ¿Cómo se conectaron los 5 compañeros?

1. Un compañero creó la base de datos en Render y obtuvo las credenciales de conexión.
2. Se creó un **usuario PostgreSQL** por cada integrante del equipo con sus propias credenciales.
3. Cada integrante configuró **pgAdmin 4** en su ordenador registrando un nuevo servidor con los datos de Render.
4. Al conectarse, todos apuntan a la **misma base de datos en la nube**  de ahí que se vean los mismos datos y los 5 usuarios activos simultáneamente.

### Evidencia de la conexión

![descripción](assets/erd_pgadmin.png)

---
## Conclusiones

El diseño de esta base de datos sigue los principios de **normalización hasta la 3FN**, garantizando:

- **Integridad referencial**: todas las relaciones están correctamente enlazadas mediante claves foráneas.
- **Sin redundancia**: los atributos descriptivos (nombre de campus, vertical, modalidad, rol) se almacenan una única vez en sus tablas propias.
- **Flexibilidad**: el rol del profesor es dinámico por promoción, permitiendo que un mismo profesor evolucione entre roles a lo largo del tiempo.
- **Trazabilidad**: cada nota queda vinculada a un alumno y un proyecto específicos mediante la tabla `calificacion`.
- **Trabajo colaborativo real**: la base de datos fue desplegada en la nube mediante Render, permitiendo que los 5 integrantes del equipo se conectaran simultáneamente a través de pgAdmin 4 y trabajaran sobre el mismo entorno de forma coordinada.

---
*Trabajo grupal — TheBridge Data Science y Full Stack Bootcamp — 2026*